# Preparing the Data for Training

This notebook takes the validated, stratified dataset produced in
`01_data_generation.ipynb` (2,350 samples, split 80/10/10 into train/validation/test)
and converts it into the exact tensor format the multi-task model expects: tokenized
input, BIO tags aligned to individual subword tokens, and multi-hot domain vectors.

Three requirements shaped the choices below:
- **Multilingual descriptions** — samples are a roughly balanced mix of French and
  English, so tokenization can't assume a single language.
- **Technology names as first-class signal** — tech names (PostgreSQL, TensorFlow,
  Node.js, ...) need to tokenize in a way that preserves their identity even when
  split into subwords.
- **Token-level BIO tags** — entity spans are annotated at the character level in the
  raw data and need to be re-aligned to whichever tokenizer is chosen.

## 1. Tokenizer

Two approaches were considered:
- Two separate tokenizers, one per language — but this requires knowing (or detecting)
  the input language before tokenization, adding a dependency the pipeline shouldn't need.
- A single cross-lingual pretrained tokenizer, avoiding language detection entirely and
  handling code-mixed text (a French sentence containing English tech names) naturally.

The second approach was chosen: `xlm-roberta-base`, pretrained on 100+ languages with a
shared SentencePiece vocabulary.

In [2]:
from transformers import AutoTokenizer, AutoModel
import json

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

DATASET_PATH = "../data/cleaned/dataset_corrected.jsonl"

In [3]:
with open(DATASET_PATH,"r",encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

### Inspecting a Sample Record

Before tokenizing the full dataset, we trace a single record end-to-end to confirm the
format matches expectations.

In [8]:
text = data[0].get("text")
text

"Design and Dev Collab Certificate. Awarded the 'Design and Dev Collab' certificate for participating in a collaborative hackathon focused on bridging UI/UX design and frontend development. Contributed as a Frontend Developer, implementing responsive interfaces using React and modern web technologies while collaborating closely with designers to deliver a functional and user-centered solution under tight deadlines."

In [10]:
tokens = tokenizer.tokenize(text)
print(tokens)

['▁Design', '▁and', '▁Dev', '▁Col', 'lab', '▁Certifica', 'te', '.', '▁Award', 'ed', '▁the', "▁'", 'Design', '▁and', '▁Dev', '▁Col', 'lab', "'", '▁certificate', '▁for', '▁participat', 'ing', '▁in', '▁a', '▁collabora', 'tive', '▁hack', 'a', 'thon', '▁focused', '▁on', '▁', 'brid', 'ging', '▁UI', '/', 'UX', '▁design', '▁and', '▁front', 'end', '▁development', '.', '▁Con', 'tribute', 'd', '▁as', '▁a', '▁Front', 'end', '▁Developer', ',', '▁implement', 'ing', '▁responsive', '▁interface', 's', '▁using', '▁Re', 'act', '▁and', '▁modern', '▁web', '▁technologies', '▁while', '▁collabora', 'ting', '▁close', 'ly', '▁with', '▁designer', 's', '▁to', '▁deliver', '▁a', '▁functional', '▁and', '▁user', '-', 'center', 'ed', '▁solution', '▁under', '▁tight', '▁deadline', 's', '.']


XLM-RoBERTa uses SentencePiece subword tokenization: each word is broken into smaller
units, improving generalization to rare or complex tokens across languages that share a
vocabulary — exactly the situation for technology names like `PostgreSQL` or
`TensorFlow`, which rarely appear as single tokens in any general-purpose vocabulary.

The `▁` character (SentencePiece's "start of word" marker, U+2581 — not an underscore)
prefixes the first subword of each new word, letting the tokenizer reconstruct original
word boundaries from a flat token stream.

In [11]:
encoded = tokenizer(text)
print(encoded)

{'input_ids': [0, 11724, 136, 40317, 11554, 6114, 83991, 67, 5, 60992, 297, 70, 242, 128640, 136, 40317, 11554, 6114, 25, 163769, 100, 42938, 214, 23, 10, 57119, 4935, 85526, 11, 50828, 162393, 98, 6, 100269, 9966, 111481, 64, 71609, 4331, 136, 12912, 3611, 34754, 5, 1657, 191145, 71, 237, 10, 43643, 3611, 152774, 4, 29479, 214, 226874, 101758, 7, 17368, 853, 47013, 136, 5744, 1467, 118299, 12960, 57119, 1916, 20903, 538, 678, 51517, 7, 47, 75060, 10, 123309, 136, 38937, 9, 30090, 297, 29806, 1379, 107137, 157206, 7, 5, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


This is the conversion of human-readable text into the tensors the model expects:
numeric vocabulary IDs corresponding to each token, and an attention mask indicating
which positions are real tokens versus padding.

For a single example like this, the attention mask is all 1s, every position is real
content. Once samples are batched together with padding, padded positions will be
marked 0 so the model's attention mechanism ignores them.

In [12]:
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print(tokens)

['<s>', '▁Design', '▁and', '▁Dev', '▁Col', 'lab', '▁Certifica', 'te', '.', '▁Award', 'ed', '▁the', "▁'", 'Design', '▁and', '▁Dev', '▁Col', 'lab', "'", '▁certificate', '▁for', '▁participat', 'ing', '▁in', '▁a', '▁collabora', 'tive', '▁hack', 'a', 'thon', '▁focused', '▁on', '▁', 'brid', 'ging', '▁UI', '/', 'UX', '▁design', '▁and', '▁front', 'end', '▁development', '.', '▁Con', 'tribute', 'd', '▁as', '▁a', '▁Front', 'end', '▁Developer', ',', '▁implement', 'ing', '▁responsive', '▁interface', 's', '▁using', '▁Re', 'act', '▁and', '▁modern', '▁web', '▁technologies', '▁while', '▁collabora', 'ting', '▁close', 'ly', '▁with', '▁designer', 's', '▁to', '▁deliver', '▁a', '▁functional', '▁and', '▁user', '-', 'center', 'ed', '▁solution', '▁under', '▁tight', '▁deadline', 's', '.', '</s>']


`<s>` marks the start of sequence and `</s>` marks the end — both required by
RoBERTa-family models, visible as the first and last entries of `input_ids` above.

To validate this behavior across vocabulary the model will actually see in production,
we test tokenization on a representative set of technology names:

In [13]:
samples = [
    "TensorFlow",
    "PyTorch",
    "React.js",
    "Node.js",
    "PostgreSQL",
    "FastAPI",
    "AWS Lambda",
    "Docker Compose",
    "Kubernetes"
]

for s in samples:
    print("\n", s)
    print(tokenizer.tokenize(s))


 TensorFlow
['▁Ten', 'sor', 'F', 'low']

 PyTorch
['▁Py', 'Tor', 'ch']

 React.js
['▁Re', 'act', '.', 'js']

 Node.js
['▁No', 'de', '.', 'js']

 PostgreSQL
['▁Post', 'gre', 'SQL']

 FastAPI
['▁Fast', 'API']

 AWS Lambda
['▁A', 'WS', '▁Lamb', 'da']

 Docker Compose
['▁Do', 'cker', '▁Comp', 'ose']

 Kubernetes
['▁Kub', 'erne', 'tes']


This demonstrates XLM-R's subword tokenization on domain-specific vocabulary:
`PostgreSQL` splits into `Post`/`gre`/`SQL`, keeping `SQL` as an isolated, reusable
subword the model has seen frequently elsewhere. `Node.js` splits into
`No`/`de`/`.`/`js`, similarly isolating `js` as its own token rather than burying it
inside a rare compound word.

In [14]:
text_fr = "Développement d'une application avec React et Python"

tokens_fr = tokenizer.tokenize(text_fr)
print(tokens_fr)

['▁D', 'ével', 'opp', 'ement', '▁d', "'", 'une', '▁application', '▁avec', '▁Re', 'act', '▁et', '▁Python']


The tokenizer segments French text into linguistically coherent subwords without any
language flag or separate vocabulary — confirming `xlm-roberta-base`'s shared
multilingual vocabulary handles French as well as English at the tokenization level.
(Semantic understanding is a separate, later claim, evaluated through the trained
model's F1 scores — clean tokenization is a necessary precondition for it, not proof of it.)

In [15]:
text = "React with Node.js"

enc = tokenizer(
    text,
    return_offsets_mapping=True
)

for tok_id, offset in zip(enc["input_ids"], enc["offset_mapping"]):
    token = tokenizer.convert_ids_to_tokens(tok_id)
    print(token, offset)

<s> (0, 0)
▁Re (0, 2)
act (2, 5)
▁with (6, 10)
▁No (11, 13)
de (13, 15)
. (15, 16)
js (16, 18)
</s> (0, 0)


These character-level offsets are exactly what's needed to convert the raw,
character-indexed entity annotations from `01_data_generation.ipynb` into token-level
BIO tags: each entity's `(start, end)` span can be mapped onto the tokens whose offsets
fall inside it.

In [4]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from scripts.preprocess import align_ner_labels, encode_domains, TAG2ID, ID2TAG, DOMAIN2ID, MAX_LENGTH

sample = data[0]
aligned = align_ner_labels(
    text=sample["text"],
    entities=sample["entities"],
    tokenizer=tokenizer,
    tag2id=TAG2ID,
    max_length=MAX_LENGTH,
)

for tok, label_id in zip(aligned["tokens"], aligned["ner_labels"]):
    tag = ID2TAG.get(label_id, "IGNORE")
    print(f"{tok:15s} {tag}")

domain_vector = encode_domains(sample["domain_labels"], DOMAIN2ID)
print("\nOriginal domains:", sample["domain_labels"])
print("Domain vector:   ", domain_vector)

<s>             IGNORE
▁Design         O
▁and            O
▁Dev            O
▁Col            O
lab             O
▁Certifica      O
te              O
.               O
▁Award          O
ed              O
▁the            O
▁'              O
Design          O
▁and            O
▁Dev            O
▁Col            O
lab             O
'               O
▁certificate    O
▁for            O
▁participat     O
ing             O
▁in             O
▁a              O
▁collabora      O
tive            O
▁hack           O
a               O
thon            O
▁focused        O
▁on             O
▁               O
brid            O
ging            O
▁UI             O
/               O
UX              O
▁design         O
▁and            O
▁front          O
end             O
▁development    O
.               O
▁Con            O
tribute         O
d               O
▁as             O
▁a              O
▁Front          O
end             O
▁Developer      O
,               O
▁implement      O
ing             O
▁resp

### From Demonstration to Full Pipeline

The full pipeline — applying this alignment across every split, encoding all ten
domains, and writing `train_processed.jsonl` / `val_processed.jsonl` /
`test_processed.jsonl` — is run via `scripts/preprocess.py`, which also persists the
tag/domain ID mappings to `data/cleaned/processed/mappings.json` for reproducibility.